# Dataset Regressão - Wind Power Generation

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [14]:
# carregamento inicial dos dados
df = pd.read_csv("../data/raw/wind-power-generation.csv", encoding='utf-8')

print(df.head())
print(df.shape)

                  Time  temperature_2m  relativehumidity_2m  dewpoint_2m  \
0  2017-01-02 00:00:00            28.5                   85         24.5   
1  2017-01-02 01:00:00            28.4                   86         24.7   
2  2017-01-02 02:00:00            26.8                   91         24.5   
3  2017-01-02 03:00:00            27.4                   88         24.3   
4  2017-01-02 04:00:00            27.3                   88         24.1   

   windspeed_10m  windspeed_100m  winddirection_10m  winddirection_100m  \
0           1.44            1.26                146                 162   
1           2.06            3.99                151                 158   
2           1.30            2.78                148                 150   
3           1.30            2.69                 58                 105   
4           2.47            4.43                 58                  84   

   windgusts_10m   Power  
0            1.4  0.1635  
1            4.4  0.1424  
2          

In [15]:
# visualização das informações principais do dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43800 entries, 0 to 43799
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Time                 43800 non-null  object 
 1   temperature_2m       43800 non-null  float64
 2   relativehumidity_2m  43800 non-null  int64  
 3   dewpoint_2m          43800 non-null  float64
 4   windspeed_10m        43800 non-null  float64
 5   windspeed_100m       43800 non-null  float64
 6   winddirection_10m    43800 non-null  int64  
 7   winddirection_100m   43800 non-null  int64  
 8   windgusts_10m        43800 non-null  float64
 9   Power                43800 non-null  float64
dtypes: float64(6), int64(3), object(1)
memory usage: 3.3+ MB


In [16]:
# validando a existencia valores ausentes ou falsos zeros
df.isnull().sum()

Time                   0
temperature_2m         0
relativehumidity_2m    0
dewpoint_2m            0
windspeed_10m          0
windspeed_100m         0
winddirection_10m      0
winddirection_100m     0
windgusts_10m          0
Power                  0
dtype: int64

In [17]:

df['hour'] = pd.to_datetime(df['Time']).dt.hour

for col in ['winddirection_10m', 'winddirection_100m']:
    rad = np.deg2rad(df[col])
    df[col + '_sin'] = np.sin(rad)
    df[col + '_cos'] = np.cos(rad)

df = df.drop(columns=['Time','winddirection_10m','winddirection_100m'])
df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=['Power'])
y = df['Power']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

param_lr = {
    # Nenhum hiperparâmetro relevante no LinearRegression do sklearn
}


In [20]:
pipe_rf = RandomForestRegressor(random_state=42)

param_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}


In [21]:
pipe_gb = GradientBoostingRegressor(random_state=42)

param_gb = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.4],
    'max_depth': [2, 3, 4, 5]
}


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grids = {
    "Linear Regression": GridSearchCV(pipe_lr, param_lr, cv=cv, scoring='r2', n_jobs=-1),
    "Random Forest": GridSearchCV(pipe_rf, param_rf, cv=cv, scoring='r2', n_jobs=-1),
    "Gradient Boosting": GridSearchCV(pipe_gb, param_gb, cv=cv, scoring='r2', n_jobs=-1)
}

best_models = {}


In [23]:
for name, grid in grids.items():
    print(f"Treinando {name}...")
    grid.fit(X_train, y_train)
    best_models[name] = grid


Treinando Linear Regression...
Treinando Gradient Boosting...


In [24]:
from math import sqrt

results = {}

for name, grid in best_models.items():
    best = grid.best_estimator_
    preds = best.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    rmse = sqrt(mean_squared_error(y_test, preds))
    
    results[name] = {
        'Melhor R2 (teste)': r2,
        'Melhor RMSE (teste)': rmse,
        'Melhores hiperparâmetros': grid.best_params_
    }

results


{'Linear Regression': {'Melhor R2 (teste)': 0.6250817514018718,
  'Melhor RMSE (teste)': 0.17614032781883032,
  'Melhores hiperparâmetros': {}},
 'Gradient Boosting': {'Melhor R2 (teste)': 0.7260353861248524,
  'Melhor RMSE (teste)': 0.15056975588673513,
  'Melhores hiperparâmetros': {'learning_rate': 0.1,
   'max_depth': 5,
   'n_estimators': 400}}}